# GraphGrasp — Training on Google Colab

**Model:** GCN_CAM_8_8_16_16_32
**Dataset:** HOGraspNet — 28 classes, no collapse (Baseline run: diagnostic for taxonomy experiment)

**Setup checklist:**
- Runtime set to GPU: `Runtime > Change runtime type > T4 GPU`
- CSV uploaded to Drive:

```
MyDrive/AIST-hand/data/raw/
  hograspnet.csv
```

- GitHub Personal Access Token (repo scope) ready to paste in step 2.

> **Note (Baseline):** This run trains on all 28 HOGraspNet classes with no collapse.
> The resulting confusion matrix cross-referenced with the synergy space analysis
> (experiments/taxonomy_v1) determines which classes to collapse for the final taxonomy.
> Processed graph cache suffix: `hograspnet_{split}_c28_cmc.pt`.
> Expect ~20-30 min for graph construction on first run.

## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU found. Go to Runtime > Change runtime type and select T4 GPU.')

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

Adjust the paths if your files live somewhere else in Drive.

In [ ]:
import os

DRIVE_DATA_DIR   = '/content/drive/MyDrive/AIST-hand/data/raw'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/AIST-hand/experiments'
GITHUB_TOKEN     = ''  # paste your token here
GITHUB_USER      = 'isapedraza'
REPO_NAME        = 'AIST-hand'

fname = 'hograspnet.csv'
path = os.path.join(DRIVE_DATA_DIR, fname)
exists = os.path.exists(path)
size_mb = os.path.getsize(path) / 1e6 if exists else 0
print(f"  {'OK' if exists else 'MISSING':8s} {fname} ({size_mb:.0f} MB)")
if not exists:
    raise FileNotFoundError(f'{fname} not found in {DRIVE_DATA_DIR}')

## 3. Clone the repository

In [ ]:
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    os.system(f'git clone {clone_url} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')
    print('Already cloned, pulled latest.')

## 4. Install dependencies

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')

!pip install -q torch-geometric
!pip install -q scikit-learn tensorboard pandas numpy
!pip install -q -e /content/AIST-hand/grasp-model

print('Done.')

## 5. Link data from Drive

Symlinks to avoid copying the CSVs (~1 GB) into the Colab disk.

In [ ]:
raw_dir       = f'{REPO_DIR}/grasp-model/data/raw'
processed_dir = f'{REPO_DIR}/grasp-model/data/processed'
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

src = os.path.join(DRIVE_DATA_DIR, 'hograspnet.csv')
dst = os.path.join(raw_dir, 'hograspnet.csv')
if not os.path.exists(dst):
    os.symlink(src, dst)
print(f'  {dst}')

## 5b. Restore processed graph cache from Drive (optional)

If you already ran this before, restore the `.pt` files from Drive to skip the 20–30 min graph conversion.

In [ ]:
DRIVE_PROCESSED = '/content/drive/MyDrive/AIST-hand/data/processed'

if os.path.exists(DRIVE_PROCESSED):
    restored = 0
    for fname in os.listdir(DRIVE_PROCESSED):
        src = os.path.join(DRIVE_PROCESSED, fname)
        dst = os.path.join(processed_dir, fname)
        if not os.path.exists(dst):
            import shutil
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(dst) / 1e6
            print(f'  Restored: {fname} ({size_mb:.0f} MB)')
            restored += 1
    if restored == 0:
        print('  Nothing new to restore (already up to date or Drive folder empty).')
else:
    print('  No cache on Drive yet — will be generated during training (step 6).')

## 6. Train

The first run converts the CSVs to PyG graphs and caches them as `.pt` files (~20–30 min for 1.4 M samples). After that, runs start from the cache.

In [ ]:
os.chdir(f'{REPO_DIR}/grasp-model')
print(f'cwd: {os.getcwd()}')

In [ ]:
!python train.py

## 7. Save results to Drive

In [ ]:
import shutil

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

src = f'{REPO_DIR}/grasp-model/experiments/best_model.pth'
dst = os.path.join(DRIVE_OUTPUT_DIR, 'best_model.pth')
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f'Saved: {dst}')
else:
    print(f'Not found: {src}')

tb_src = f'{REPO_DIR}/grasp-model/experiments/runs'
tb_dst = os.path.join(DRIVE_OUTPUT_DIR, 'runs')
if os.path.exists(tb_src):
    if os.path.exists(tb_dst):
        shutil.rmtree(tb_dst)
    shutil.copytree(tb_src, tb_dst)
    print(f'TensorBoard logs saved: {tb_dst}')

## 8. TensorBoard (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/AIST-hand/grasp-model/experiments/runs

## 9. Cache processed graphs to Drive (optional)

Save the `.pt` files so the next session skips the conversion step.

In [ ]:
DRIVE_PROCESSED = '/content/drive/MyDrive/AIST-hand/data/processed'
os.makedirs(DRIVE_PROCESSED, exist_ok=True)

local_processed = f'{REPO_DIR}/grasp-model/data/processed'
for fname in os.listdir(local_processed):
    src = os.path.join(local_processed, fname)
    dst = os.path.join(DRIVE_PROCESSED, fname)
    shutil.copy2(src, dst)
    size_mb = os.path.getsize(dst) / 1e6
    print(f'  {fname} ({size_mb:.0f} MB)')